# MAT322 Group 3: Monte Carlo simulation
Buying now (deposit plus mortgage) versus investing first and buying later, with the same monthly budget for both households.

This notebook is the simulation part of the report (Part B). It follows Section 4.5 and equations (10) to (12) of the combined draft.

The setup is:
1. The buy-now household puts R1.5m down, takes a R2m mortgage and pays the monthly instalment M. Its house grows at a fixed rate g.
2. The buy-later household invests the R1.5m and also invests the same amount M every month, so both households spend the same each month.
3. The investment return is random. We simulate 10,000 possible futures and count how often the buy-later household ends up with more wealth than the buy-now household.

To run it, use Runtime > Run all. All the assumptions are in the inputs cell, so change them there only.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from decimal import Decimal, ROUND_HALF_UP

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.options.display.float_format = "{:,.2f}".format

def pct(x, n=10_000):
    # percentage to 1 decimal place, rounding .x5 upwards
    cnt = Decimal(int(round(x * n)))
    return str((cnt / Decimal(n) * 100).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)) + "%"

This cell loads the libraries. The small function `pct` turns a probability into a percentage with one decimal place. We wrote it because Python's built-in formatting rounds a value like 91.25% down to 91.2%, while the usual convention (and what the report uses) is 91.3%. Without it the notebook and the report would disagree in a few cells.

## 1. Inputs
These are the group's master assumptions from Table 1 of the combined draft. Items 1 to 9 come from the draft. Item 10, volatility, is different: the Satrix 40 Minimum Disclosure Document does not give one, so 16% is a working assumption for a large South African equity fund. Sections 6a and 6b below show how to check it.

In [7]:
P0     = 3_500_000    # starting house price (R)
C      = 1_500_000    # starting capital (R)
L      = P0 - C       # mortgage amount (R2,000,000)
rm     = 0.1075       # mortgage rate per year (same for Absa and Nedbank)
N      = 240          # number of monthly payments (20 years)
g      = 0.08         # house price growth per year (9%, 10% and 11% are tested later)
cagr   = 0.1238       # Satrix 40 ten year annualised return at 31 Aug 2026
infl   = 0.052        # inflation per year
horizons = [10, 15, 20]

sigma  = 0.16         # annual volatility, an assumption (see sections 6a and 6b)

n_sims = 10_000       # number of simulated paths, the draft asks for at least 10,000
seed   = 2026         # fixed so the results can be reproduced
months = 240

Everything in the notebook reads from this cell. `L` is just the price minus the deposit. `seed` makes the random numbers repeatable, so anyone who reruns the notebook gets the same tables as the report.

## 2. Mortgage and house price (no randomness)
This part uses the formulas from the draft: the monthly payment (equation 4), the balance left after k payments (equation 5) and the house value after t years (equation 2). Equity is the house value minus the balance still owing.

In [8]:
def payment(L, rm, N):
    i = rm / 12
    return L * i / (1 - (1 + i) ** -N)

def balance(L, rm, N, k):
    i = rm / 12
    M = payment(L, rm, N)
    return L * (1 + i) ** k - M * ((1 + i) ** k - 1) / i

def house_value(P0, g, t):
    return P0 * (1 + g) ** t

M = payment(L, rm, N)
print(f"Monthly instalment M = R{M:,.2f}")

det = []
for t in horizons:
    H = house_value(P0, g, t)
    B = balance(L, rm, N, 12 * t)
    det.append({"Horizon (yrs)": t, "House value H": H, "Mortgage balance": B, "Buy-now equity E": H - B})
det = pd.DataFrame(det).set_index("Horizon (yrs)")
display(det)

assert abs(M - 20304.58) < 0.01
assert abs(det.loc[10, "Buy-now equity E"] - 6_066_962.47) < 0.05
assert abs(det.loc[15, "Buy-now equity E"] - 10_163_345.78) < 0.05
print("Matches the draft's Tables 3 and 6.")

Monthly instalment M = R20,304.58


,House value H,Mortgage balance,Buy-now equity E
Horizon (yrs),,,
10,"7,556,237.49","1,489,275.02","6,066,962.47"
15,"11,102,591.90","939,246.12","10,163,345.78"
20,"16,313,350.00",0.00,"16,313,350.00"


Matches the draft's Tables 3 and 6.


The instalment comes out at R20,304.58 and the balances and equity agree with the draft, so the deterministic side is right. The `assert` lines stop the notebook if any of these numbers is off. They only hold for the base inputs, so delete them if you change an input on purpose.

The buy-now household's equity is the number the investing household has to beat, and it does not depend on the random part at all.

## 3. The simulation
Each month the investment is multiplied by a random growth factor and then the month-end instalment M is added. The growth factor follows geometric Brownian motion (equation 12 of the draft), stepped monthly.

On calibration: 12.38% is a compound (geometric) return, so it cannot be used directly as the drift of the model. We set the average monthly log growth to ln(1.1238)/12. This makes the median lump-sum outcome equal the plain 12.38% calculation, and the mean comes out higher, which is the right relationship between a geometric and an arithmetic average.

In [9]:
def simulate(Z, cagr, sigma, M, C, months=240):
    # V: investment with monthly contributions. lump: the R1.5m alone with no contributions.
    n = Z.shape[0]
    growth = np.exp(np.log(1 + cagr) / 12 + sigma / np.sqrt(12) * Z)
    V = np.empty((n, months + 1)); V[:, 0] = C
    lump = np.empty((n, months + 1)); lump[:, 0] = C
    for k in range(months):
        V[:, k + 1] = V[:, k] * growth[:, k] + M
        lump[:, k + 1] = lump[:, k] * growth[:, k]
    return V, lump

rng = np.random.default_rng(seed)
Z = rng.standard_normal((n_sims, months))   # the same random numbers are reused in every scenario below

# check: with sigma = 0 there is no randomness, so the result must equal the closed form formulas
V0, lump0 = simulate(Z, cagr, 0.0, M, C)
j = (1 + cagr) ** (1 / 12) - 1
for t in horizons:
    k = 12 * t
    closed = C * (1 + cagr) ** t + M * ((1 + j) ** k - 1) / j
    assert abs(V0[0, k] - closed) < 1e-4
    assert abs(lump0[0, k] - C * (1 + cagr) ** t) < 1e-4
print("sigma = 0 reproduces the deterministic formulas exactly.")

sigma = 0 reproduces the deterministic formulas exactly.


`Z` holds 10,000 rows of 240 standard normal random numbers, one row per simulated future and one column per month. We generate it once and reuse it for every scenario, so when a table changes it is because the assumption changed and not because of different random numbers.

The `simulate` function returns two sets of paths. `V` is the equal-budget investor, who adds M each month. `lump` is the R1.5m on its own, which we use to compare with Part A.

The check at the end sets sigma to zero. With no randomness the simulation has to give exactly the same answer as the closed form formulas in Part A, and it does, which tells us the code is doing what the maths says.

## 4. Base case results
For each horizon we report the mean, median, standard deviation, 5th and 95th percentiles, the share of paths that can pay for the house, and the share of paths where investing first beats buying now. These are the items listed in Section 4.5 of the draft.

In [10]:
V, lump = simulate(Z, cagr, sigma, M, C, months)

rows = []
for t in horizons:
    k = 12 * t
    H = house_value(P0, g, t)
    E = H - balance(L, rm, N, k)
    v, l = V[:, k], lump[:, k]
    rows.append({
        "Horizon (yrs)": t,
        "Mean": v.mean(), "Median": np.median(v), "Std dev": v.std(),
        "5th pct": np.percentile(v, 5), "95th pct": np.percentile(v, 95),
        "P(funds house, equal budget)": np.mean(v >= H),
        "P(invest-first > buy-now)": np.mean(v > E),
        "P(lump sum alone funds house)": np.mean(l >= H),
    })
res = pd.DataFrame(rows).set_index("Horizon (yrs)")

money = ["Mean", "Median", "Std dev", "5th pct", "95th pct"]
pcts = [c for c in res.columns if c.startswith("P(")]
display(res.style.format({**{c: "R{:,.0f}" for c in money}, **{c: (lambda v: pct(v, n_sims)) for c in pcts}}))
print(f"n_sims = {n_sims:,}, seed = {seed}, sigma = {sigma:.0%} (assumed), cagr = {cagr:.2%}, g = {g:.0%}")

AttributeError: The '.style' accessor requires jinja2

Each row is one horizon, taken from column 12t of the simulated paths (month 120, 180 or 240). The probabilities are just proportions: for example "invest-first > buy-now" is the fraction of the 10,000 paths where the investor's wealth ends above the buyer's equity.

Two things to notice. The mean is higher than the median at every horizon, because returns compound multiplicatively and a few very good paths pull the average up. And the lump sum on its own rarely pays for the house (47% of paths at 20 years), so the high equal-budget probabilities come mostly from the monthly contributions, not from the R1.5m.

In [ ]:
# fan chart: investor's wealth (percentile bands) against the buyer's equity, on a log scale
yrs = np.arange(months + 1) / 12
k_arr = np.arange(months + 1)
H_path = P0 * (1 + g) ** yrs
B_path = np.array([balance(L, rm, N, k) if k <= N else 0.0 for k in k_arr])
E_path = H_path - B_path

fig, ax = plt.subplots()
p5, p25, p50, p75, p95 = [np.percentile(V, q, axis=0) / 1e6 for q in (5, 25, 50, 75, 95)]
ax.fill_between(yrs, p5, p95, alpha=0.20, color="#2ca02c", label="Invest-first: 5th to 95th percentile")
ax.fill_between(yrs, p25, p75, alpha=0.35, color="#2ca02c", label="Invest-first: 25th to 75th percentile")
ax.plot(yrs, p50, color="#2ca02c", lw=2, label="Invest-first: median")
ax.plot(yrs, E_path / 1e6, color="#1f77b4", lw=2.5, label="Buy-now: housing equity")
ax.plot(yrs, H_path / 1e6, color="gray", lw=1.5, ls="--", label="Property price")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R{x:,.0f}m"))
ax.set_xlabel("Years"); ax.set_title("Equal monthly budget: invest-first wealth vs buy-now equity")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.savefig("mc_fan_chart.png", dpi=150); plt.show()

The green bands show where 90% (light) and 50% (dark) of the simulated investor outcomes fall over time, and the green line is the median. The blue line is the buyer's equity. The vertical axis is on a log scale so the early years are readable. Where the blue line sits inside or below the green band, the investor is usually ahead. The band gets very wide by year 20, which is the uncertainty the deterministic Part A could not show.

In [ ]:
# histograms of the investor's wealth at each horizon
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, t in zip(axes, horizons):
    k = 12 * t
    H = house_value(P0, g, t); E = H - balance(L, rm, N, k)
    v = V[:, k] / 1e6
    ax.hist(v[v <= np.percentile(v, 99)], bins=60, color="#2ca02c", alpha=0.75)
    ax.axvline(E / 1e6, color="#1f77b4", lw=2, ls="--", label=f"Buy-now equity R{E/1e6:,.1f}m")
    ax.axvline(H / 1e6, color="gray", lw=1.5, ls=":", label=f"House price R{H/1e6:,.1f}m")
    ax.set_title(f"{t} years: P(invest-first > buy-now) = {pct(np.mean(V[:, k] > E), n_sims)}")
    ax.set_xlabel("Invest-first wealth (R millions)"); ax.legend(fontsize=8)
axes[0].set_ylabel("Number of simulations")
plt.tight_layout(); plt.savefig("mc_histograms.png", dpi=150); plt.show()

These show the full spread of investor wealth at years 10, 15 and 20. The dashed blue line is the buyer's equity, so everything to the right of it is a path where investing first wins. The histograms drop the top 1% of paths so the extreme outliers do not squash the chart. The distribution is skewed to the right, which matches the mean being above the median.

## 5. Sensitivity: which assumption matters most?
The draft asks which assumptions drive the answer. The next cells change one assumption at a time and reuse the same random numbers, so the tables can be compared fairly.

In [ ]:
# investment return (columns) against volatility (rows); each cell is P(invest-first beats buy-now)
cagr_grid  = [0.06, 0.08, 0.10, 0.12, 0.1238, 0.14]
sigma_grid = [0.10, 0.14, 0.18, 0.22]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, t in zip(axes, horizons):
    k = 12 * t
    E = house_value(P0, g, t) - balance(L, rm, N, k)
    grid = np.zeros((len(sigma_grid), len(cagr_grid)))
    for a, s in enumerate(sigma_grid):
        for b, c in enumerate(cagr_grid):
            Vs, _ = simulate(Z, c, s, M, C, months)
            grid[a, b] = np.mean(Vs[:, k] > E)
    ax.imshow(grid, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(cagr_grid))); ax.set_xticklabels([f"{c:.2%}" for c in cagr_grid], fontsize=8)
    ax.set_yticks(range(len(sigma_grid))); ax.set_yticklabels([f"{s:.0%}" for s in sigma_grid])
    ax.set_xlabel("Annual investment return (compound)"); ax.set_ylabel("Volatility")
    ax.set_title(f"{t} years"); ax.grid(False)
    for a in range(len(sigma_grid)):
        for b in range(len(cagr_grid)):
            ax.text(b, a, pct(grid[a, b], n_sims)[:-1], ha="center", va="center", fontsize=8)
fig.suptitle("P(invest-first beats buy-now), equal monthly budget", y=1.02)
plt.tight_layout(); plt.savefig("mc_heatmaps.png", dpi=150, bbox_inches="tight"); plt.show()

This runs the full simulation 24 times for each horizon (6 returns by 4 volatilities) and colours each cell by the probability that investing first wins. Green means investing first usually wins and red means it usually loses. The numbers in the cells are the percentages used in Tables B4 to B6 of the report.

Reading across a row, the probability rises quickly with the assumed return. Reading down a column, it falls only a little as volatility goes up. So the return assumption matters much more than the volatility one, which is why the 16% assumption is not a big weakness.

In [ ]:
# property growth: the investment paths stay the same, only the target (house price and equity) moves
rows = []
for gg in [0.08, 0.09, 0.10, 0.11]:
    r = {"Property growth g": f"{gg:.0%}"}
    for t in horizons:
        k = 12 * t
        H = house_value(P0, gg, t); E = H - balance(L, rm, N, k)
        r[f"{t}y P(> buy-now)"] = np.mean(V[:, k] > E)
        r[f"{t}y P(funds house)"] = np.mean(V[:, k] >= H)
    rows.append(r)
tabB = pd.DataFrame(rows).set_index("Property growth g")
display(tabB.style.format(lambda v: pct(v, n_sims)))

# mortgage rate: this changes the instalment M (so the investor's monthly contribution) and the buyer's equity
rows = []
for rr in [0.0975, 0.1025, 0.1075, 0.1125, 0.1175]:
    Mx = payment(L, rr, N)
    Vx, _ = simulate(Z, cagr, sigma, Mx, C, months)
    r = {"Mortgage rate": f"{rr:.2%}", "Instalment M": f"R{Mx:,.0f}"}
    for t in horizons:
        k = 12 * t
        E = house_value(P0, g, t) - balance(L, rr, N, k)
        r[f"{t}y P(> buy-now)"] = pct(np.mean(Vx[:, k] > E), n_sims)
    rows.append(r)
display(pd.DataFrame(rows).set_index("Mortgage rate"))

The first table changes the house price growth. Faster growth means a more expensive house and more equity for the buyer, so the probability that investing first wins falls from about 87% at 8% growth to about 61% at 11% (10 year horizon). Property growth is the second most important assumption after the investment return.

The second table changes the mortgage rate, and the result looks backwards: a higher mortgage rate makes investing first look slightly better. This is a side effect of how the comparison is built. A higher rate means a bigger instalment M, and the investing household puts that whole larger amount into the fund each month while paying no rent. So the table reflects the modelling choice (equal budget, rent excluded) and is not evidence that expensive mortgages favour investing. The report says this explicitly.

## 6. Today's money
Here we divide the terminal values by 1.052^t to express them in today's rand, as in Section 3.6 of the draft. This only adjusts for the fall in purchasing power. It is not a risk-adjusted discount rate.

In [ ]:
rows = []
for t in horizons:
    k = 12 * t
    E = house_value(P0, g, t) - balance(L, rm, N, k)
    d = (1 + infl) ** t
    rows.append({"Horizon (yrs)": t,
                 "Buy-now equity (today's R)": E / d,
                 "Invest-first median (today's R)": np.median(V[:, k]) / d,
                 "Invest-first 5th pct (today's R)": np.percentile(V[:, k], 5) / d})
display(pd.DataFrame(rows).set_index("Horizon (yrs)").style.format("R{:,.0f}"))

In today's money the median investor is well ahead of the buyer, but the 5th percentile (a bad outcome that happens about 1 time in 20) is below the buyer's equity at every horizon. So investing first usually wins, but when it goes badly the investor ends up with less than the buyer.

## 7. How stable are the probabilities?
A Monte Carlo answer has sampling error because it comes from a finite number of paths. With 10,000 paths the standard error of a probability is at most 0.5 percentage points (about 0.3 near 87%). The cell below repeats the base case with seven other seeds to check this in practice.

In [ ]:
rows = []
for sd in [1, 2, 3, 4, 5, 11, 99]:
    Zs = np.random.default_rng(sd).standard_normal((n_sims, months))
    Vs, ls = simulate(Zs, cagr, sigma, M, C, months)
    r = {"Seed": sd}
    for t in horizons:
        k = 12 * t
        E = house_value(P0, g, t) - balance(L, rm, N, k)
        r[f"{t}y P(> buy-now)"] = np.mean(Vs[:, k] > E)
    rows.append(r)
seedtab = pd.DataFrame(rows).set_index("Seed")
display(seedtab.style.format(lambda v: pct(v, n_sims)))
print("Range across seeds:")
for c in seedtab.columns:
    print(f"  {c}: {pct(seedtab[c].min(), n_sims)} to {pct(seedtab[c].max(), n_sims)}")

Changing the seed moves the base case probabilities by about one percentage point at most. So a difference of one point or less between two cells in the report tables is just noise and should not be read as meaningful.

## 8. Checking the volatility assumption
### 8a. Rough check from the Satrix 40 disclosure
The Minimum Disclosure Document (31 August 2026) reports a highest annual rolling return of 27.06% and a lowest of -2.98%, from 10 non-overlapping one-year periods. For 10 independent normal observations the expected range is about 3.078 standard deviations, so range divided by 3.078 gives a rough estimate of sigma.

In [ ]:
hi, lo, d10 = 27.06, -2.98, 3.078
sigma_range = (hi - lo) / d10 / 100
print(f"Range-based volatility estimate: {sigma_range:.1%}  (assumption used in the model: {sigma:.0%})")

This gives about 9.8%, lower than the 16% we assumed. It only uses the two most extreme years, so it is a sanity check and not a proper estimate. It does tell us that 16% is not too low for this ten year window, and the sensitivity tables already cover volatilities from 10% to 22%.

### 8b. Proper estimate from monthly prices
If a CSV of monthly closing prices for the Satrix 40 ETF (STX40) over about September 2016 to August 2026 is available (for example from Investing.com or Yahoo Finance), set the three variables below and run the cell. It needs a date column and a price column. Then copy the printed volatility into the inputs cell and rerun the notebook.

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

csv_path = Path(
    r"C:\Users\melod\Downloads\MAT322 PROJECT\STX40J Historical Data (1).csv"
)

date_col, price_col = "Date", "Price"

px = pd.read_csv(csv_path)

print("File exists:", csv_path.exists())
print("Shape:", px.shape)
print("Columns:", px.columns.tolist())

px[date_col] = pd.to_datetime(px[date_col])
px = px.sort_values(date_col)

px[price_col] = (
    px[price_col]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

logret = np.log(px[price_col]).diff().dropna()
sigma_monthly = logret.std(ddof=1)
sigma_annual = sigma_monthly * np.sqrt(12)

print(
    f"Observations: {len(logret)} "
    f"({px[date_col].iloc[0].date()} to {px[date_col].iloc[-1].date()})"
)
print(f"Monthly log return standard deviation: {sigma_monthly:.4f}")
print(f"Annualised volatility (sigma): {sigma_annual:.2%}")
print(f"Mean log return per year: {logret.mean() * 12:.2%}")

File exists: True
Shape: (130, 7)
Columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
Observations: 129 (2016-01-01 to 2026-10-01)
Monthly log return standard deviation: 0.0422
Annualised volatility (sigma): 14.61%
Mean log return per year: 8.17%


The volatility is the standard deviation of the monthly log returns, multiplied by the square root of 12 to turn it into a yearly figure. A closing price series ignores dividends, but dividends add very little to volatility, so this is a fair estimate. If we use it, the report should say that the data was price only.

## 9. Limits of the model (for the report)
1. Volatility is an assumption (16%), not a measured value. See section 8.
2. Rent is excluded. The investing household is treated as living for free for up to 20 years while it invests every instalment, which favours investing first a lot. Adding even a rough rent would change the comparison.
3. The result depends mainly on the assumed 12.38% return, which is a strong ten year result compared with a 10.75% mortgage rate. It is not a safe long-run figure.
4. No tax is applied, although the proposal mentions after-tax returns. The 8% and 10% columns of the sensitivity table can be read as rough after-tax cases.
5. Property growth is fixed, so every probability is conditional on the chosen growth rate.
6. Investor wealth V is compared with buyer equity E. If V is at least the house price H the investor pays cash and keeps V - H, otherwise the shortfall is financed and net worth is still V before costs. Transaction costs are excluded.
7. The report should state n_sims, the seed, sigma and the model used (geometric Brownian motion, monthly steps).